## PACOTES ##

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import json
from pathlib import Path
from IPython.display import display

## CÓDIGO ##

In [ ]:
from pathlib import Path

BASE_PATH = Path("thoracic_surgery_base_modelo_statsmodels.csv")
OUTPUT_DIR = Path("outputs")

OUTPUT_DIR.mkdir(exist_ok=True)

VAR_RESPOSTA = "obito_1_ano"

In [ ]:

dados = pd.read_csv(BASE_PATH)

print("Arquivo utilizado:")
print(BASE_PATH)

print("\nDimensão da base:")
print(dados.shape)

print("\nPrimeiras linhas da base:")
display(dados.head())

print("\nColunas da base:")
display(pd.DataFrame({"colunas": dados.columns}))

Arquivo utilizado:
thoracic_surgery_base_modelo_statsmodels.csv

Dimensão da base:
(470, 26)

Primeiras linhas da base:


,obito_1_ano,constante,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN1]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN2]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN4]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN5]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN6]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN8]","C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ0]","C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ2]",...,hemoptise_antes_cirurgia,dispneia_antes_cirurgia,tosse_antes_cirurgia,fraqueza_antes_cirurgia,diabetes_mellitus_tipo_2,infarto_miocardio_ate_6_meses,doenca_arterial_periferica,tabagismo,asma,idade
0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,60.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,51.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,59.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.0
4,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,73.0



Colunas da base:


,colunas
0,obito_1_ano
1,constante
2,"C(diagnostico, Treatment(reference='DGN3'))[T...."
3,"C(diagnostico, Treatment(reference='DGN3'))[T...."
4,"C(diagnostico, Treatment(reference='DGN3'))[T...."
5,"C(diagnostico, Treatment(reference='DGN3'))[T...."
6,"C(diagnostico, Treatment(reference='DGN3'))[T...."
7,"C(diagnostico, Treatment(reference='DGN3'))[T...."
8,"C(estado_desempenho_zubrod, Treatment(referenc..."
9,"C(estado_desempenho_zubrod, Treatment(referenc..."


In [ ]:

if VAR_RESPOSTA not in dados.columns:
    raise ValueError(f"A variável resposta '{VAR_RESPOSTA}' não foi encontrada na base.")


dados[VAR_RESPOSTA] = pd.to_numeric(dados[VAR_RESPOSTA], errors="coerce")

n_ausentes_resposta = dados[VAR_RESPOSTA].isna().sum()

if n_ausentes_resposta > 0:
    raise ValueError(
        f"A variável resposta possui {n_ausentes_resposta} valores ausentes ou não numéricos."
    )

valores_resposta = sorted(dados[VAR_RESPOSTA].unique())

print("Valores encontrados na variável resposta:")
print(valores_resposta)


if set(valores_resposta) != {0, 1}:
    raise ValueError(
        "A variável resposta deve conter apenas os valores 0 e 1. "
        f"Valores encontrados: {valores_resposta}"
    )

frequencia_resposta = (
    dados[VAR_RESPOSTA]
    .value_counts()
    .sort_index()
    .rename(index={0: "Não óbito", 1: "Óbito"})
    .to_frame("frequência")
)

percentual_resposta = (
    dados[VAR_RESPOSTA]
    .value_counts(normalize=True)
    .sort_index()
    .rename(index={0: "Não óbito", 1: "Óbito"}) * 100
).round(2).to_frame("porcentagem")

print("\nDistribuição da variável resposta:")
display(frequencia_resposta)

print("\nDistribuição percentual da variável resposta:")
display(percentual_resposta)

print("\nResumo:")
print(f"Total de observações: {len(dados)}")
print(f"Número de óbitos em até 1 ano: {int(dados[VAR_RESPOSTA].sum())}")
print(f"Número de não óbitos: {int((1 - dados[VAR_RESPOSTA]).sum())}")
print(f"Proporção de óbitos: {dados[VAR_RESPOSTA].mean() * 100:.2f}%")

Valores encontrados na variável resposta:
[np.float64(0.0), np.float64(1.0)]

Distribuição da variável resposta:


,frequência
obito_1_ano,
Não óbito,400
Óbito,70



Distribuição percentual da variável resposta:


,porcentagem
obito_1_ano,
Não óbito,85.11
Óbito,14.89



Resumo:
Total de observações: 470
Número de óbitos em até 1 ano: 70
Número de não óbitos: 400
Proporção de óbitos: 14.89%


In [ ]:

y = dados[VAR_RESPOSTA].astype(float)
X = dados.drop(columns=[VAR_RESPOSTA]).copy()


for coluna in X.columns:
    X[coluna] = pd.to_numeric(X[coluna], errors="coerce")

faltantes = X.isna().sum()
faltantes = faltantes[faltantes > 0]

if len(faltantes) > 0:
    print("Covariáveis com valores ausentes:")
    display(faltantes.to_frame("n_faltantes"))
    raise ValueError("Existem valores ausentes nas covariáveis. Corrija antes de ajustar o modelo.")


colunas_constantes = []

for coluna in X.columns:
    if X[coluna].nunique() == 1:
        colunas_constantes.append(coluna)

print("Colunas constantes encontradas:")
print(colunas_constantes)

if len(colunas_constantes) == 0:
    print("\nNenhuma coluna constante foi encontrada. Será adicionada uma constante ao modelo.")
    X = sm.add_constant(X, has_constant="add")
else:
    print("\nA base já possui coluna constante. Nenhuma constante adicional será criada.")

print("\nDimensão da variável resposta y:")
print(y.shape)

print("\nDimensão da matriz de covariáveis X:")
print(X.shape)

print("\nVariáveis que entrarão no modelo completo:")
display(pd.DataFrame({"variavel": X.columns}))

print("\nPrimeiras linhas da matriz X:")
display(X.head())

Colunas constantes encontradas:
['constante']

A base já possui coluna constante. Nenhuma constante adicional será criada.

Dimensão da variável resposta y:
(470,)

Dimensão da matriz de covariáveis X:
(470, 25)

Variáveis que entrarão no modelo completo:


,variavel
0,constante
1,"C(diagnostico, Treatment(reference='DGN3'))[T...."
2,"C(diagnostico, Treatment(reference='DGN3'))[T...."
3,"C(diagnostico, Treatment(reference='DGN3'))[T...."
4,"C(diagnostico, Treatment(reference='DGN3'))[T...."
5,"C(diagnostico, Treatment(reference='DGN3'))[T...."
6,"C(diagnostico, Treatment(reference='DGN3'))[T...."
7,"C(estado_desempenho_zubrod, Treatment(referenc..."
8,"C(estado_desempenho_zubrod, Treatment(referenc..."
9,"C(tamanho_tumor_tnm, Treatment(reference='OC11..."



Primeiras linhas da matriz X:


,constante,"C(diagnostico, Treatment(reference='DGN3'))[T.DGN1]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN2]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN4]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN5]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN6]","C(diagnostico, Treatment(reference='DGN3'))[T.DGN8]","C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ0]","C(estado_desempenho_zubrod, Treatment(reference='PRZ1'))[T.PRZ2]","C(tamanho_tumor_tnm, Treatment(reference='OC11'))[T.OC12]",...,hemoptise_antes_cirurgia,dispneia_antes_cirurgia,tosse_antes_cirurgia,fraqueza_antes_cirurgia,diabetes_mellitus_tipo_2,infarto_miocardio_ate_6_meses,doenca_arterial_periferica,tabagismo,asma,idade
0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,60.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,51.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,59.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,73.0


In [11]:
modelo_completo = sm.GLM(
    y,
    X,
    family=sm.families.Binomial()
)

resultado_completo = modelo_completo.fit(maxiter=200)

print(resultado_completo.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            obito_1_ano   No. Observations:                  470
Model:                            GLM   Df Residuals:                      445
Model Family:                Binomial   Df Model:                           24
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -170.59
Date:                Thu, 18 Jun 2026   Deviance:                       341.19
Time:                        23:20:11   Pearson chi2:                     418.
No. Iterations:                    21   Pseudo R-squ. (CS):             0.1093
Covariance Type:            nonrobust                                         
                                                                       coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------

In [ ]:

tabela_coeficientes = pd.DataFrame({
    "variavel": resultado_completo.params.index,
    "coeficiente": resultado_completo.params.values,
    "erro_padrao": resultado_completo.bse.values,
    "estatistica_wald_z": resultado_completo.tvalues.values,
    "p_valor": resultado_completo.pvalues.values
})

tabela_coeficientes = tabela_coeficientes.sort_values(
    by="p_valor",
    ascending=False
).reset_index(drop=True)

display(tabela_coeficientes)

,variavel,coeficiente,erro_padrao,estatistica_wald_z,p_valor
0,"C(diagnostico, Treatment(reference='DGN3'))[T....",-20.180552,48196.142312,-0.000419,0.999666
1,asma,-19.983295,33047.010568,-0.000605,0.999518
2,infarto_miocardio_ate_6_meses,-20.655378,33212.257674,-0.000622,0.999504
3,"C(diagnostico, Treatment(reference='DGN3'))[T....",-19.771699,23658.138605,-0.000836,0.999333
4,doenca_arterial_periferica,-0.097894,1.003314,-0.097571,0.922273
5,"C(estado_desempenho_zubrod, Treatment(referenc...",0.149014,0.578271,0.257689,0.796647
6,hemoptise_antes_cirurgia,0.174337,0.389186,0.447952,0.654188
7,idade,-0.009506,0.018099,-0.525204,0.599442
8,"C(estado_desempenho_zubrod, Treatment(referenc...",0.442715,0.519908,0.851525,0.394478
9,"C(diagnostico, Treatment(reference='DGN3'))[T....",0.427777,0.473330,0.903761,0.366122


In [13]:
# Informações principais do modelo completo

info_modelo_completo = pd.DataFrame({
    "medida": [
        "n_observacoes",
        "n_eventos_obito",
        "n_nao_eventos",
        "numero_parametros",
        "deviance",
        "graus_liberdade_residuo",
        "log_verossimilhanca"
    ],
    "valor": [
        int(resultado_completo.nobs),
        int(y.sum()),
        int((1 - y).sum()),
        int(len(resultado_completo.params)),
        float(resultado_completo.deviance),
        int(resultado_completo.df_resid),
        float(resultado_completo.llf)
    ]
})

display(info_modelo_completo)

,medida,valor
0,n_observacoes,470.000000
1,n_eventos_obito,70.000000
2,n_nao_eventos,400.000000
3,numero_parametros,25.000000
4,deviance,341.187221
5,graus_liberdade_residuo,445.000000
6,log_verossimilhanca,-170.593610


In [ ]:

caminho_modelo_completo = OUTPUT_DIR / "modelo_completo.pkl"
caminho_variaveis_modelo = OUTPUT_DIR / "variaveis_modelo.json"

resultado_completo.save(str(caminho_modelo_completo), remove_data=False)

informacoes_variaveis = {
    "arquivo_base": str(BASE_PATH),
    "variavel_resposta": VAR_RESPOSTA,
    "n_observacoes": int(resultado_completo.nobs),
    "n_eventos_obito": int(y.sum()),
    "n_nao_eventos": int((1 - y).sum()),
    "variaveis_modelo": X.columns.tolist(),
    "variaveis_explicativas_modelo": [
        col for col in X.columns if col not in ["const", "constante"]
    ],
    "colunas_constantes": colunas_constantes,
    "deviance_modelo_completo": float(resultado_completo.deviance),
    "graus_liberdade_residuo": int(resultado_completo.df_resid),
    "log_verossimilhanca": float(resultado_completo.llf)
}

with open(caminho_variaveis_modelo, "w", encoding="utf-8") as arquivo:
    json.dump(informacoes_variaveis, arquivo, ensure_ascii=False, indent=4)

print("Arquivos salvos com sucesso:")
print(f"- {caminho_modelo_completo}")
print(f"- {caminho_variaveis_modelo}")

Arquivos salvos com sucesso:
- outputs\modelo_completo.pkl
- outputs\variaveis_modelo.json
